In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.read_csv("../data/VidProm_unique.csv")
df['prompt_length'] = df['prompt'].str.len()
df['word_count'] = df['prompt'].str.split().str.len()
df['time'] = pd.to_datetime(df['time'], format='%a %b %d %H:%M:%S %Y')
df.shape

(1672243, 11)

In [3]:
nsfw_cols = ['toxicity', 'obscene', 'identity_attack', 'insult', 'threat', 'sexual_explicit']

stats = {
    "total_prompts": len(df),
    "date_from": str(df['time'].min()),
    "date_to": str(df['time'].max()),
    "prompt_length": {
        "mean": round(df['prompt_length'].mean(), 1),
        "median": int(df['prompt_length'].median()),
    },
    "word_count": {
        "mean": round(df['word_count'].mean(), 1),
        "median": int(df['word_count'].median()),
    },
    "nsfw_above_05": {
        col: round((df[col] > 0.5).mean() * 100, 2)
        for col in nsfw_cols
    }
}

with open('../data/stats_overview.json', 'w') as f:
    json.dump(stats, f, indent=4)

    print(json.dumps(stats, indent=2))

{
  "total_prompts": 1672243,
  "date_from": "2023-06-29 08:20:41",
  "date_to": "2024-03-03 18:03:41",
  "prompt_length": {
    "mean": 118.6,
    "median": 68
  },
  "word_count": {
    "mean": 18.5,
    "median": 11
  },
  "nsfw_above_05": {
    "toxicity": 1.89,
    "obscene": 0.06,
    "identity_attack": 0.14,
    "insult": 0.57,
    "threat": 0.11,
    "sexual_explicit": 0.16
  }
}


In [4]:
stop_words = {'a', 'an', 'the', 'of', 'in', 'on', 'at', 'to', 'and', 'is', 'it',
                'for', 'with', 'that', 'this', 'are', 'was', 'be', 'as', 'by', 'or',
                'from', 'but', 'not', 'its', 'has', 'have', 'had', 'been', 'will',
                'can', 'do', 'does', 'did', 'he', 'she', 'they', 'we', 'you', 'i',
                'my', 'his', 'her', 'their', 'our', 'your', 'no', 'so', 'if', 'about'}

all_words = df['prompt'].str.lower().str.findall(r'[a-z]+').explode()
filtered = all_words[~all_words.isin(stop_words)]
top30 = filtered.value_counts().head(30)

words_list = [{"word": w, "count": int(c)} for w, c in top30.items()]

with open('../data/top_words.json', 'w') as f:
    json.dump(words_list, f, indent=2)

    print(words_list[:5])

[{'word': 'ar', 'count': 223257}, {'word': 'motion', 'count': 213235}, {'word': 's', 'count': 175111}, {'word': 'quot', 'count': 174687}, {'word': 'camera', 'count': 163017}]


In [5]:
cluster_names = {
      0: "Discord junk",
      1: "Epic/historical scenes",
      2: "Animals/creatures",
      3: "Cinematic/aesthetic",
      4: "Mixed/general",
      5: "Women/girls",
      6: "Men/characters",
      7: "Horror/dark",
      8: "Animation/cartoon",
      9: "Nature/landscapes"
}

sample = pd.read_csv('../data/sample_clustered.csv')

clusters = []
for cid in sorted(sample['cluster'].unique()):
    group = sample[sample['cluster'] == cid]
    clusters.append({
        "id": int(cid),
        "name": cluster_names[cid],
        "count": len(group),
        "sample_prompts": group['prompt'].sample(5, random_state=42).tolist()
    })

with open('../data/clusters_summary.json', 'w') as f:
    json.dump(clusters, f, indent=2)

print(f"saved {len(clusters)} clusters")
for c in clusters:
    print(f"  Cluster {c['id']} ({c['name']}): {c['count']} prompts")

saved 10 clusters
  Cluster 0 (Discord junk): 3924 prompts
  Cluster 1 (Epic/historical scenes): 9553 prompts
  Cluster 2 (Animals/creatures): 10132 prompts
  Cluster 3 (Cinematic/aesthetic): 12691 prompts
  Cluster 4 (Mixed/general): 15700 prompts
  Cluster 5 (Women/girls): 8929 prompts
  Cluster 6 (Men/characters): 9636 prompts
  Cluster 7 (Horror/dark): 7478 prompts
  Cluster 8 (Animation/cartoon): 10694 prompts
  Cluster 9 (Nature/landscapes): 11263 prompts
